# 03 Features (feature set v3)

**Purpose.** Build one feature table per split: one row per (search, candidate) pair with the label and all model inputs. Features are organised in **groups** so that `05_models` can run controlled ablations (PROJECT_PLAN section 5) and measure what each kind of information adds.

v2 replaced v1 (E07) after two independent reviews (E08): popularity encoded so training and validation rows have the same distribution, missing text treated as unknown, reproducible shuffling, renamed blocks, and history groups restructured so that the window and context questions can be tested separately.

**v3 (E13), after the data-generation audits (E12):** `n_candidates` removed (list length reflects how far the user scrolled, i.e. it forms after the query); a deployable history popularity `pop_history_lag1d` (only history actions at least one day before their owner's search) separated from same-day popularity; metadata coverage per history block added, so a model can tell "no match" from "nothing observable". Searches whose final gap is clamped to 1 second (post-search actions in the history, E12) are not removed here: they are flagged in the search table and excluded from training and the main evaluation downstream.

| Group | Features | Question it serves |
|---|---|---|
| **G0 search** | query length | base (constant within a list) |
| **G1 relevance** | query-title shared terms, coverage, Jaccard, BM25, title length; query term in brand / L3 category name; candidate in catalogue | base, non-personal |
| **G2a popularity, deployable** | history popularity from actions at least one day old | base, non-personal, deployable |
| **G2b popularity, same day** | exposures, unlagged history popularity (all other training searches), out-of-fold smoothed engagement rate | base, non-personal; not available at serving time |
| **H1 pooled history** | same product, same shop (other products), same brand (other products), same L3 path, over all history | Team 3-like history |
| **H2 history by type** | the same four matches per action type (CLICK, CART, ORD, FLW), any age | type without age |
| **H3 history by window** | the same four matches per block (0-7 d CLICK / CART / ORD, 7+ d CART, 7-90 d ORD, 90+ d ORD, FLW) | type and age (observation windows) |
| **H4 match recency** | days since the latest same product / shop / brand action | age of the matching evidence |
| **C1 history volume** | history length, actions per block | context carried by volume |
| **H5 metadata coverage** | share of actions per block whose product has a catalogue row | observability of broad matches |
| **C2 session context** | days since the last action, actions in the last 7 days, share of query-less history, type of the last action (categorical) | session context |
| **Q1 query-history** | max Jaccard current vs past queries, any overlap; product / shop-other / brand-other matches from history whose query shares a term with the current one, all ages and last 7 days | query relevance of history |
| **S1 history source** | product / shop-other / brand-other matches from search-linked vs query-less history | active vs passive intent |

**Rules.**
- Candidates are shuffled within each search with a seed derived from the search ID (`default_rng([4222, session_id])`), so the order does not depend on which searches are built. No feature uses a position.
- Missing metadata gives `NaN` ("unknown", not "no match"), separately per field: a missing title only blanks the title features, a missing brand name only `query_in_brand`. Category matches use the full ancestor path.
- **Popularity (G2), training records only.** Exposures and history counts use no labels: training rows count all *other* training searches (their own search excluded), validation and test rows count all training searches. The engagement rate uses labels, so it is **out-of-fold**: every search gets a fold (seeded per search ID), and its rate, including the prior, is computed from the other four folds only. Validation and test searches also get a fold and are encoded the same way, so both have identical encoding distributions. Rate = (engaged + 20 x prior) / (exposures + 20).
- **Same-day caveat:** all searches happen on 17 Oct 2022, so popularity uses other users' outcomes from the same day. A deployed system would not have them; the non-personalised reference is therefore optimistic and personalisation gains measured on top of it are conservative.
- BM25 (k1 = 1.2, b = 0.75) uses document frequencies and average title length over the whole catalogue (no labels involved).
- `MIN_AGE_DAYS` (default 0) drops history actions younger than this before all history and context features: the robustness check with a safety gap before the search (PROJECT_PLAN section 5).

**Splits.** Locally: `train_dev` (the 40,000-search development subsample) and `validation`. On Colab (high-RAM runtime) additionally `train_rest` (the other 93,831 training searches, for the final models trained on all training data) and `test` (scored once, only for the final models). Every feature of a search is independent of which other searches are built, so the splits can be built in separate runs.

**Outputs** (Git-ignored `data/processed/` locally, `MyDrive/BT4222 Group Project/processed/` on Colab): `features_v3_{split}.parquet` and `features_v3_meta.json` (groups, categorical and search-level features).

**Run time:** about 30-35 minutes locally for `train_dev` and `validation` (behaviour passes ~2 min, catalogue ~4 min, features ~15 min); all four splits on Colab about 1-1.5 hours.

**How to run on Colab (all splits).** Copy `split_v1.csv` and `search_table.parquet` (written by notebook 02) to `MyDrive/BT4222 Group Project/processed/`; open this notebook from GitHub (branch `BRANCH` in the first code cell); choose a high-RAM CPU runtime and *Run all*. The first cell clones the public repository and mounts Drive; outputs go to the Drive folder above. Save the executed notebook (*File > Download > .ipynb*) so its outputs can be committed.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import math
import sys
import time

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "feat/models"  # branch to clone on Colab; "main" once merged
REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None and IN_COLAB:  # opened from GitHub: clone the (public) repository, or update an earlier clone
    import subprocess
    REPO_ROOT = Path("/content/BT4222-group17-jdsearch")
    if (REPO_ROOT / ".git").is_dir():  # a second run in the same runtime: bring the clone to the branch's latest commit
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                        "https://github.com/liamkunert/BT4222-group17-jdsearch.git", str(REPO_ROOT)], check=True)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
    PROCESSED = Path("/content/drive/MyDrive/BT4222 Group Project/processed")
else:
    DATA_DIR = REPO_ROOT / "data"
    PROCESSED = REPO_ROOT / "data/processed"
BEHAVIOUR, CATALOGUE = DATA_DIR / "user_behavior_data.txt", DATA_DIR / "product_meta_data.txt"

sys.path.insert(0, str(REPO_ROOT))
from scripts.jd_data import iter_records, SECONDS_PER_DAY
from scripts.evaluation import evaluate
from scripts.history_match_diagnostics import metadata_keys, validate_match_helpers
from scripts.query_diagnostics import overlap_features, token_set, tokenize, validate_helpers

validate_helpers()
validate_match_helpers()
SEED, N_FOLDS, PRIOR_STRENGTH = 4222, 5, 20
K1, B = 1.2, 0.75
MIN_AGE_DAYS = 0.0  # > 0 only for the history safety-gap robustness check
VERSION = "v3" if MIN_AGE_DAYS == 0 else f"v3_minage{MIN_AGE_DAYS:g}d"
DAY = SECONDS_PER_DAY
pd.options.display.float_format = "{:,.4f}".format


def split_terms(text):
    # Catalogue texts: plain split on the term separator (no validation, so one malformed title cannot stop the scan).
    return text.split("\x18") if text and text != "-1" else []


def fold_of(session_id):
    # Fold per search, seeded by its ID: independent of file order and of which searches are built.
    return int(np.random.default_rng([SEED, 7, session_id]).integers(N_FOLDS))


assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
SPLITS = {
    "train_dev": assignment.index[assignment.dev_subsample],
    "validation": assignment.index[assignment.split == "validation"],
    "train_rest": assignment.index[(assignment.split == "train") & ~assignment.dev_subsample],  # final models only
    "test": assignment.index[assignment.split == "test"],  # scored once, for the final models only
}
# All four need a high-RAM runtime (about 174,000 searches in memory); locally only the dev subsample and validation.
SPLITS_TO_BUILD = list(SPLITS) if IN_COLAB else ["train_dev", "validation"]
BUILD = {name: SPLITS[name].to_numpy() for name in SPLITS_TO_BUILD}
train_ids = set(assignment.index[assignment.split == "train"].tolist())
assert all(sid in train_ids for sid in BUILD["train_dev"])
folds = {sid: fold_of(sid) for sid in assignment.index}
print({name: len(ids) for name, ids in BUILD.items()}, "| version", VERSION,
      "| fold sizes in train:", np.bincount([folds[s] for s in train_ids]))

Mounted at /content/drive
{'train_dev': 40000, 'validation': 20000, 'train_rest': 93831, 'test': 20000} | version v3 | fold sizes in train: [26946 26816 26554 26938 26577]


## 1. Load the target searches

Candidates are shuffled within each search with a per-search seed. History younger than `MIN_AGE_DAYS` is dropped here (no effect at the default 0).

In [2]:
target = {int(sid): name for name, ids in BUILD.items() for sid in ids}
records = []
for session_id, fields in iter_records(BEHAVIOUR):
    if session_id not in target:
        continue
    candidates = np.array(fields[1].split("_"))
    labels = np.array([int(float(v)) for v in fields[2].split("_")], dtype=np.int8)
    order = np.random.default_rng([SEED, session_id]).permutation(len(candidates))
    gaps = np.array(fields[6].split("_"), dtype=np.int64)
    elapsed_d = np.cumsum(gaps[::-1])[::-1][1:] / DAY
    keep = elapsed_d >= MIN_AGE_DAYS
    past, queries = np.array(fields[4].split("_")), np.array(fields[3].split("_"))
    records.append(dict(session_id=session_id, split=target[session_id], query=fields[0],
                        candidates=candidates[order], labels=labels[order], past=past[keep].tolist(), past_all=past.tolist(), elapsed_all=elapsed_d,
                        types=np.array(fields[5].split("_"))[keep], queries=queries[keep].tolist(),
                        elapsed_d=elapsed_d[keep]))
candidate_wids = {w for r in records for w in r["candidates"]}
history_wids = {w for r in records for w in r["past"]}
query_vocab = set().union(*(token_set(r["query"]) for r in records))
print(f"searches {len(records):,}; distinct candidates {len(candidate_wids):,}; "
      f"distinct history products {len(history_wids):,}; query terms {len(query_vocab):,}; "
      f"searches with empty history after MIN_AGE: {sum(len(r['types']) == 0 for r in records)}")

searches 173,831; distinct candidates 8,362,535; distinct history products 5,584,543; query terms 47,461; searches with empty history after MIN_AGE: 0


## 2. Catalogue pass

Metadata (brand, shop, category path) for candidates and history products; raw title, brand name and L3 category name for candidates; BM25 statistics over all titles.

In [3]:
start = time.time()
needed = candidate_wids | history_wids
metadata, names = {}, {}
doc_freq, n_docs, total_len = Counter(), 0, 0
with CATALOGUE.open(encoding="utf-8") as handle:
    assert next(handle).split("\t", 1)[0] == "wid"
    for line in handle:
        wid, title, rest = line.split("\t", 2)
        terms = split_terms(title)
        if terms:
            n_docs += 1
            total_len += len(terms)
            doc_freq.update(query_vocab.intersection(terms))
        if wid in needed:
            fields = line.rstrip("\r\n").split("\t")
            metadata[wid] = tuple(int(fields[j]) if fields[j] not in ("", "-1") else None for j in (2, 12, 4, 6, 8, 10))
            if wid in candidate_wids:
                names[wid] = (fields[1], fields[3], fields[9])
avg_title_len = total_len / n_docs
idf = {t: math.log(1 + (n_docs - doc_freq[t] + 0.5) / (doc_freq[t] + 0.5)) for t in query_vocab}
missing_text = pd.Series({field: np.mean([not split_terms(v[i]) for v in names.values()])
                          for i, field in enumerate(["title", "brand_name", "cate3_name"])})
print(f"catalogue titles {n_docs:,}, average length {avg_title_len:.1f}; metadata for "
      f"{len(metadata) / len(needed):.1%} of needed products; {time.time() - start:.0f} s")
print("share of candidates (with a catalogue row) whose text field is missing:")
print(missing_text.to_string())

catalogue titles 12,141,247, average length 27.9; metadata for 94.1% of needed products; 302 s
share of candidates (with a catalogue row) whose text field is missing:
title        0.0000
brand_name   0.0891
cate3_name   0.0000


## 3. Training-only popularity (G2)

One pass over all training records. Exposures and history counts are kept as totals (no labels involved); training rows subtract their own search later. Exposures and engagements are also kept per fold for the out-of-fold engagement rate, together with the per-fold totals for the prior.

In [4]:
item_index = {w: i for i, w in enumerate(sorted(candidate_wids))}
exposures = np.zeros((N_FOLDS, len(item_index)), np.int32)
engaged = np.zeros((N_FOLDS, len(item_index)), np.int32)
in_history = np.zeros(len(item_index), np.int32)
in_history_lag = np.zeros(len(item_index), np.int32)  # actions at least 1 day before their owner's search
fold_engaged, fold_shown = np.zeros(N_FOLDS), np.zeros(N_FOLDS)
train_dev_queries = {r["query"] for r in records if r["split"] == "train_dev"}
same_query_lists = defaultdict(list)  # for the mechanism check in section 6
for session_id, fields in iter_records(BEHAVIOUR):
    if session_id not in train_ids:
        continue
    fold = folds[session_id]
    wids = fields[1].split("_")
    labels = [int(float(v)) for v in fields[2].split("_")]
    fold_engaged[fold] += sum(l >= 1 for l in labels)
    fold_shown[fold] += len(labels)
    for wid, label in zip(wids, labels):
        i = item_index.get(wid)
        if i is not None:
            exposures[fold, i] += 1
            engaged[fold, i] += label >= 1
    gaps = np.array(fields[6].split("_"), dtype=np.int64)
    ages = np.cumsum(gaps[::-1])[::-1][1:] / DAY
    for wid, age in zip(fields[4].split("_"), ages):
        i = item_index.get(wid)
        if i is not None:
            in_history[i] += 1
            in_history_lag[i] += age >= 1
    if fields[0] in train_dev_queries:
        same_query_lists[fields[0]].append((session_id, wids))
EXP_TOTAL, ENG_TOTAL = exposures.sum(axis=0), engaged.sum(axis=0)
print("training engagement rate per fold:", np.round(fold_engaged / fold_shown, 5))


def popularity(r):
    wids = r["candidates"]
    idx = np.array([item_index[w] for w in wids])
    exp, hist = EXP_TOTAL[idx].astype(float), in_history[idx].astype(float)
    hist_lag = in_history_lag[idx].astype(float)
    if r["session_id"] in train_ids:  # training rows: exclude the own list and history from the label-free counts
        in_list = Counter(wids)
        exp -= np.array([in_list[w] for w in wids])
        own = Counter(r["past_all"])  # unfiltered: in_history counts full histories
        hist -= np.array([own[w] for w in wids])
        own_lag = Counter(w for w, a in zip(r["past_all"], r["elapsed_all"]) if a >= 1)
        hist_lag -= np.array([own_lag[w] for w in wids])
    fold = folds[r["session_id"]]
    oof_exp = EXP_TOTAL[idx] - exposures[fold, idx]
    oof_eng = ENG_TOTAL[idx] - engaged[fold, idx]
    prior = (fold_engaged.sum() - fold_engaged[fold]) / (fold_shown.sum() - fold_shown[fold])
    rate = (oof_eng + PRIOR_STRENGTH * prior) / (oof_exp + PRIOR_STRENGTH)
    return exp, rate, hist, hist_lag

training engagement rate per fold: [0.02066 0.0207  0.02073 0.02058 0.0206 ]


## 4. Feature definitions

In [5]:
LEVELS = {"product": 0, "brand": 1, "shop": 2, "L3": 5}  # positions in metadata_keys
TYPES = ["CLICK", "CART", "ORD", "FLW"]
BLOCKS = {
    "click_0_7": lambda t, d: (t == "CLICK") & (d < 7),
    "cart_0_7": lambda t, d: (t == "CART") & (d < 7),
    "ord_0_7": lambda t, d: (t == "ORD") & (d < 7),
    "cart_7plus": lambda t, d: (t == "CART") & (d >= 7),
    "ord_7_90": lambda t, d: (t == "ORD") & (d >= 7) & (d < 90),
    "ord_90plus": lambda t, d: (t == "ORD") & (d >= 90),
    "flw": lambda t, d: t == "FLW",
}
LAST_TYPE = {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}
MATCHES = ["product", "shop_other", "brand_other", "L3"]


def counts(cand_keys, hist_keys, idx, pos):
    counter = Counter(hist_keys[i][pos] for i in idx)
    return np.array([counter.get(k[pos], 0) if k[pos] is not None else np.nan for k in cand_keys], dtype=float)


def matches(cand_keys, hist_keys, idx, prefix, f, which=MATCHES):
    # The four match definitions shared by all history groups: exact product, shop / brand via other products, L3 path.
    product = counts(cand_keys, hist_keys, idx, 0)
    values = {"product": product,
              "shop_other": counts(cand_keys, hist_keys, idx, 2) - product,
              "brand_other": counts(cand_keys, hist_keys, idx, 1) - product,
              "L3": counts(cand_keys, hist_keys, idx, 5)}
    for name in which:
        f[f"{prefix}_{name}"] = values[name]


def latest(cand_keys, hist_keys, elapsed_d, pos):
    newest = {}
    for keys, age in zip(hist_keys, elapsed_d):
        key = keys[pos]
        if key is not None and (key not in newest or age < newest[key]):
            newest[key] = age
    return np.array([newest.get(k[pos], np.nan) if k[pos] is not None else np.nan for k in cand_keys])


def relevance(query_terms, wid):
    # Each text field is checked separately: a missing field gives NaN only for its own features.
    if wid not in names:
        return (np.nan,) * 7
    title, brand, category = (split_terms(v) for v in names[wid])
    out = [np.nan] * 7
    if title:
        shared, coverage, jaccard = overlap_features(query_terms, frozenset(title))
        tf, length = Counter(title), len(title)
        bm25 = sum(idf[t] * tf[t] * (K1 + 1) / (tf[t] + K1 * (1 - B + B * length / avg_title_len))
                   for t in query_terms if tf[t])
        out[:5] = [shared, coverage, jaccard, bm25, length]
    if brand:
        out[5] = float(bool(query_terms.intersection(brand)))
    if category:
        out[6] = float(bool(query_terms.intersection(category)))
    return tuple(out)


def search_features(r):
    cands, n = r["candidates"], len(r["candidates"])
    types, elapsed_d = r["types"], r["elapsed_d"]
    query_terms = token_set(r["query"])
    cand_keys = [metadata_keys(w, metadata) for w in cands]
    hist_keys = [metadata_keys(w, metadata) for w in r["past"]]
    f = {"session_id": np.full(n, r["session_id"]), "wid": cands.astype(np.int64), "label": r["labels"]}
    # G0, G1, G2
    f["query_len"] = np.full(n, len(tokenize(r["query"])))
    rel = np.array([relevance(query_terms, w) for w in cands], dtype=float)
    for j, name in enumerate(["title_shared", "title_coverage", "title_jaccard", "bm25", "title_len",
                              "query_in_brand", "query_in_cate3"]):
        f[name] = rel[:, j]
    f["in_catalogue"] = np.array([w in metadata for w in cands], dtype=float)
    f["pop_exposures"], f["pop_engaged_rate"], f["pop_history"], f["pop_history_lag1d"] = popularity(r)
    # H1-H4
    matches(cand_keys, hist_keys, np.arange(len(types)), "pooled", f)
    for t in TYPES:
        matches(cand_keys, hist_keys, np.flatnonzero(types == t), f"type_{t.lower()}", f)
    block_idx = {b: np.flatnonzero(rule(types, elapsed_d)) for b, rule in BLOCKS.items()}
    for b, idx in block_idx.items():
        matches(cand_keys, hist_keys, idx, b, f)
    for level in ("product", "shop", "brand"):
        f[f"days_since_{level}"] = latest(cand_keys, hist_keys, elapsed_d, LEVELS[level])
    # C1, C2
    f["hist_len"] = np.full(n, len(types))
    for b, idx in block_idx.items():
        f[f"n_{b}"] = np.full(n, len(idx))
        f[f"cov_{b}"] = np.full(n, np.mean([r["past"][i] in metadata for i in idx]) if len(idx) else np.nan)
    f["days_since_last_action"] = np.full(n, elapsed_d[-1] if len(types) else np.nan)
    f["actions_last_7d"] = np.full(n, int((elapsed_d < 7).sum()))
    f["share_queryless"] = np.full(n, np.mean([q == "-1" for q in r["queries"]]) if len(types) else np.nan)
    f["last_action_type"] = np.full(n, LAST_TYPE[types[-1]] if len(types) else np.nan)
    # Q1
    past_terms = [token_set(q) for q in r["queries"]]
    jaccards = [len(query_terms & p) / len(query_terms | p) for p in past_terms if p]
    f["hist_query_max_jaccard"] = np.full(n, max(jaccards) if jaccards else np.nan)
    related = np.array([bool(query_terms & p) for p in past_terms], dtype=bool)
    f["hist_query_any_overlap"] = np.full(n, float(related.any()) if jaccards else np.nan)
    matches(cand_keys, hist_keys, np.flatnonzero(related), "qrel", f, MATCHES[:3])
    matches(cand_keys, hist_keys, np.flatnonzero(related & (elapsed_d < 7)), "qrel_0_7", f, MATCHES[:3])
    # S1
    linked = np.array([q != "-1" for q in r["queries"]], dtype=bool)
    matches(cand_keys, hist_keys, np.flatnonzero(linked), "linked", f, MATCHES[:3])
    matches(cand_keys, hist_keys, np.flatnonzero(~linked), "qless", f, MATCHES[:3])
    return f  # dict of per-candidate arrays (a DataFrame per search costs too much memory)


GROUPS = {
    "G0_search": ["query_len"],
    "G1_relevance": ["title_shared", "title_coverage", "title_jaccard", "bm25", "title_len", "query_in_brand",
                     "query_in_cate3", "in_catalogue"],
    "G2a_popularity_deployable": ["pop_history_lag1d"],
    "G2b_popularity_same_day": ["pop_exposures", "pop_engaged_rate", "pop_history"],
    "H1_pooled": [f"pooled_{m}" for m in MATCHES],
    "H2_by_type": [f"type_{t.lower()}_{m}" for t in TYPES for m in MATCHES],
    "H3_by_window": [f"{b}_{m}" for b in BLOCKS for m in MATCHES],
    "H4_match_recency": [f"days_since_{l}" for l in ("product", "shop", "brand")],
    "H5_metadata_coverage": [f"cov_{b}" for b in BLOCKS],
    "C1_history_volume": ["hist_len"] + [f"n_{b}" for b in BLOCKS],
    "C2_session_context": ["days_since_last_action", "actions_last_7d", "share_queryless", "last_action_type"],
    "Q1_query_history": ["hist_query_max_jaccard", "hist_query_any_overlap"]
                        + [f"qrel_{m}" for m in MATCHES[:3]] + [f"qrel_0_7_{m}" for m in MATCHES[:3]],
    "S1_history_source": [f"linked_{m}" for m in MATCHES[:3]] + [f"qless_{m}" for m in MATCHES[:3]],
}
ALL_FEATURES = [c for cols in GROUPS.values() for c in cols]
SEARCH_LEVEL = GROUPS["G0_search"] + GROUPS["H5_metadata_coverage"] + GROUPS["C1_history_volume"] + GROUPS["C2_session_context"] + \
    ["hist_query_max_jaccard", "hist_query_any_overlap"]
assert len(ALL_FEATURES) == len(set(ALL_FEATURES))
print({g: len(c) for g, c in GROUPS.items()}, "total", len(ALL_FEATURES))

{'G0_search': 1, 'G1_relevance': 8, 'G2a_popularity_deployable': 1, 'G2b_popularity_same_day': 3, 'H1_pooled': 4, 'H2_by_type': 16, 'H3_by_window': 28, 'H4_match_recency': 3, 'H5_metadata_coverage': 7, 'C1_history_volume': 8, 'C2_session_context': 4, 'Q1_query_history': 8, 'S1_history_source': 6} total 97


Toy checks: block rules at the boundaries, "other products" logic, missing-text handling.

In [6]:
_t = np.array(["CART", "CART", "ORD", "ORD", "ORD", "ORD", "CLICK"])
_d = np.array([6.999, 7.0, 89.999, 90.0, 365.0, 6.0, 7.5])
assert BLOCKS["cart_0_7"](_t, _d).tolist() == [True, False, False, False, False, False, False]
assert BLOCKS["cart_7plus"](_t, _d).tolist() == [False, True, False, False, False, False, False]
assert BLOCKS["ord_7_90"](_t, _d).tolist() == [False, False, True, False, False, False, False]
assert BLOCKS["ord_90plus"](_t, _d).tolist() == [False, False, False, True, True, False, False]
assert not any(BLOCKS[b](_t, _d)[6] for b in BLOCKS)  # a click older than 7 days belongs to no block
_meta = {"a": (10, 20, 1, 2, 3, 4), "b": (11, 20, 1, 2, 3, 5)}
_f = {}
matches([metadata_keys("a", _meta)], [metadata_keys("a", _meta), metadata_keys("b", _meta)], np.array([0, 1]), "x", _f)
assert (_f["x_product"][0], _f["x_shop_other"][0], _f["x_brand_other"][0]) == (1, 1, 0)
_term = sorted(query_vocab)[0]  # a real query term, so BM25 has its document frequency
names["_toy"] = (f"{_term}\x18999999999", "-1", "")
_rel = relevance(frozenset({_term}), "_toy")
assert np.isnan(_rel[5]) and np.isnan(_rel[6]) and _rel[0] == 1 and _rel[3] > 0
del names["_toy"]
print("feature helper checks passed")

feature helper checks passed


## 5. Build and save

In [7]:
start = time.time()
# One pre-allocated float32 block per split, filled search by search (60,000 small DataFrames ran out of memory).
n_rows = Counter()
for r in records:
    n_rows[r["split"]] += len(r["candidates"])
blocks = {s: np.empty((n, len(ALL_FEATURES)), np.float32) for s, n in n_rows.items()}
ids = {s: {"session_id": np.empty(n, np.int64), "wid": np.empty(n, np.int64), "label": np.empty(n, np.int8)}
       for s, n in n_rows.items()}
position = Counter()
for r in records:
    f, split = search_features(r), r["split"]
    lo, hi = position[split], position[split] + len(r["candidates"])
    for j, column in enumerate(ALL_FEATURES):
        blocks[split][lo:hi, j] = f[column]
    for column in ids[split]:
        ids[split][column][lo:hi] = f[column]
    position[split] = hi
assert position == n_rows
tables = {}
for split in n_rows:
    table = pd.DataFrame(blocks.pop(split), columns=ALL_FEATURES, copy=False)
    for k, column in enumerate(["session_id", "wid", "label"]):
        table.insert(k, column, ids[split][column])
    table.to_parquet(PROCESSED / f"features_{VERSION}_{split}.parquet", index=False)
    tables[split] = table
    print(f"{split}: {table.session_id.nunique():,} searches, {len(table):,} rows, {len(ALL_FEATURES)} features")
meta = {"version": VERSION, "min_age_days": MIN_AGE_DAYS, "groups": GROUPS, "categorical": ["last_action_type"],
        "search_level": SEARCH_LEVEL, "id_columns": ["session_id", "wid", "label"]}
(PROCESSED / f"features_{VERSION}_meta.json").write_text(json.dumps(meta, indent=2))
print(f"{time.time() - start:.0f} s")

train_dev: 40,000 searches, 3,573,362 rows, 97 features
train_rest: 93,831 searches, 8,370,164 rows, 97 features
validation: 20,000 searches, 1,781,573 rows, 97 features
test: 20,000 searches, 1,784,913 rows, 97 features
1088 s


## 6. Checks

1. **Completeness and consistency:** full lists; pooled product = sum over types = sum over blocks plus the few old clicks; "other product" counts never negative.
2. **Permutation test:** re-shuffling a search's candidates and recomputing gives identical features per product.
3. **Train vs validation distributions** (not just means): share of zeros, quantiles and missing share per feature, especially for popularity.
4. **Sanity against E03 and E07** (training rows only).
5. **Popularity mechanism (training rows only):** how much of the exposure signal comes from searches with the identical query, and does it hold for queries never seen in training?

In [8]:
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
for split, table in tables.items():
    sizes = table.groupby("session_id").size()
    assert (sizes == searches.loc[sizes.index, "n_candidates"]).all() and len(sizes) == len(BUILD[split])
    by_type = table[[f"type_{t.lower()}_product" for t in TYPES]].sum(axis=1)
    by_block = table[[f"{b}_product" for b in BLOCKS]].sum(axis=1)
    assert np.allclose(by_type, table.pooled_product) and (by_block <= table.pooled_product).all()
    assert (table.pooled_product - by_block).sum() < 0.001 * table.pooled_product.sum()
    other = [c for c in ALL_FEATURES if c.endswith("_other")]
    assert (table[other].fillna(0) >= 0).all().all()

check = [r for r in records[:300]]
for r in check:
    original = pd.DataFrame(search_features(r)).sort_values("wid").reset_index(drop=True)
    perm = np.random.default_rng([SEED, 99, r["session_id"]]).permutation(len(r["candidates"]))
    shuffled = pd.DataFrame(search_features({**r, "candidates": r["candidates"][perm], "labels": r["labels"][perm]}))
    shuffled = shuffled.sort_values("wid").reset_index(drop=True)
    pd.testing.assert_frame_equal(original, shuffled)
print("completeness, consistency and permutation checks passed")

completeness, consistency and permutation checks passed


In [9]:
def distribution(table):
    values = table[ALL_FEATURES]
    q = values.quantile([0.5, 0.9]).T
    return pd.DataFrame({"zero": (values == 0).mean(), "missing": values.isna().mean(),
                         "median": q[0.5], "p90": q[0.9]})


dist = pd.concat({split: distribution(table) for split, table in tables.items()}, axis=1)
dist["zero_gap"] = (dist[("validation", "zero")] - dist[("train_dev", "zero")]).abs()
dist["missing_gap"] = (dist[("validation", "missing")] - dist[("train_dev", "missing")]).abs()
for split in tables:
    if split not in ("train_dev", "validation"):  # e.g. train_rest: should match train_dev (same population)
        dist[f"zero_gap_{split}"] = (dist[(split, "zero")] - dist[("train_dev", "zero")]).abs()
        dist[f"missing_gap_{split}"] = (dist[(split, "missing")] - dist[("train_dev", "missing")]).abs()
dist.to_csv(PROCESSED / f"features_{VERSION}_distribution.csv")
print("largest zero-share gaps between training (dev) rows and each other split:")
gap_columns = [c for c in dist.columns if str(c[0]).startswith(("zero_gap", "missing_gap"))]
display(dist[gap_columns].max().rename("largest gap over all features").to_frame())
dist.sort_values("zero_gap", ascending=False).head(8)

largest zero-share gaps between training (dev) rows and each other split:


,,largest gap over all features
zero_gap,,0.0068
missing_gap,,0.0055
zero_gap_train_rest,,0.0051
missing_gap_train_rest,,0.0051
zero_gap_test,,0.0071
missing_gap_test,,0.0038


train_dev                        train_rest          \
                            zero missing median     p90       zero missing   
cart_7plus_L3             0.8309  0.0045 0.0000  2.0000     0.8275  0.0046   
type_cart_L3              0.7368  0.0045 0.0000  3.0000     0.7334  0.0046   
n_ord_0_7                 0.6246  0.0000 0.0000  3.0000     0.6235  0.0000   
hist_query_max_jaccard    0.5761  0.0267 0.0000  0.5000     0.5738  0.0259   
hist_query_any_overlap    0.5761  0.0267 0.0000  1.0000     0.5738  0.0259   
last_action_type          0.7253  0.0000 0.0000  1.0000     0.7270  0.0000   
pooled_L3                 0.5970  0.0045 0.0000 13.0000     0.5942  0.0046   
type_click_L3             0.7225  0.0045 0.0000  8.0000     0.7200  0.0046   

                                      validation          ...   test          \
                       median     p90       zero missing  ...   zero missing   
cart_7plus_L3          0.0000  2.0000     0.8241  0.0044  ... 0.8294  0.0045   
type_cart_L3           0.0000  4.0000     0.7306  0.0044  ... 0.7364  0.0045   
n_ord_0_7              0.0000  4.0000     0.6302  0.0000  ... 0.6220  0.0000   
hist_query_max_jaccard 0.0000  0.5000     0.5709  0.0248  ... 0.5690  0.0246   
hist_query_any_overlap 0.0000  1.0000     0.5709  0.0248  ... 0.5690  0.0246   
last_action_type       0.0000  1.0000     0.7300  0.0000  ... 0.7270  0.0000   
pooled_L3              0.0000 13.0000     0.5925  0.0044  ... 0.5934  0.0045   
type_click_L3          0.0000  8.0000     0.7182  0.0044  ... 0.7198  0.0045   

                                      zero_gap missing_gap  \
                       median     p90                        
cart_7plus_L3          0.0000  2.0000   0.0068      0.0001   
type_cart_L3           0.0000  3.0000   0.0061      0.0001   
n_ord_0_7              0.0000  3.0000   0.0055      0.0000   
hist_query_max_jaccard 0.0000  0.5000   0.0051      0.0019   
hist_query_any_overlap 0.0000  1.0000   0.0051      0.0019   
last_action_type       0.0000  1.0000   0.0047      0.0000   
pooled_L3              0.0000 13.0000   0.0045      0.0001   
type_click_L3          0.0000  8.0000   0.0043      0.0001   

                       zero_gap_train_rest missing_gap_train_rest  \
                                                                    
cart_7plus_L3                       0.0034                 0.0000   
type_cart_L3                        0.0034                 0.0000   
n_ord_0_7                           0.0011                 0.0000   
hist_query_max_jaccard              0.0023                 0.0008   
hist_query_any_overlap              0.0023                 0.0008   
last_action_type                    0.0017                 0.0000   
pooled_L3                           0.0028                 0.0000   
type_click_L3                       0.0024                 0.0000   

                       zero_gap_test missing_gap_test  
                                                       
cart_7plus_L3                 0.0015           0.0000  
type_cart_L3                  0.0003           0.0000  
n_ord_0_7                     0.0026           0.0000  
hist_query_max_jaccard        0.0071           0.0021  
hist_query_any_overlap        0.0071           0.0021  
last_action_type              0.0018           0.0000  
pooled_L3                     0.0036           0.0000  
type_click_L3                 0.0026           0.0000  

[8 rows x 22 columns]

In [10]:
dist.loc[GROUPS["G2a_popularity_deployable"] + GROUPS["G2b_popularity_same_day"]]

train_dev                        train_rest                 \
                       zero missing median     p90       zero missing median   
pop_history_lag1d    0.6043  0.0000 0.0000 24.0000     0.6076  0.0000 0.0000   
pop_exposures        0.4356  0.0000 1.0000 15.0000     0.4404  0.0000 1.0000   
pop_engaged_rate     0.0000  0.0000 0.0206  0.0207     0.0000  0.0000 0.0206   
pop_history          0.5911  0.0000 0.0000 28.0000     0.5945  0.0000 0.0000   

                          validation          ...   test                 \
                      p90       zero missing  ...   zero missing median   
pop_history_lag1d 24.0000     0.6072  0.0000  ... 0.6073  0.0000 0.0000   
pop_exposures     14.0000     0.4390  0.0000  ... 0.4411  0.0000 1.0000   
pop_engaged_rate   0.0207     0.0000  0.0000  ... 0.0000  0.0000 0.0206   
pop_history       27.0000     0.5940  0.0000  ... 0.5943  0.0000 0.0000   

                          zero_gap missing_gap zero_gap_train_rest  \
                      p90                                            
pop_history_lag1d 24.0000   0.0029      0.0000              0.0033   
pop_exposures     14.0000   0.0034      0.0000              0.0048   
pop_engaged_rate   0.0207   0.0000      0.0000              0.0000   
pop_history       27.0000   0.0028      0.0000              0.0034   

                  missing_gap_train_rest zero_gap_test missing_gap_test  
                                                                         
pop_history_lag1d                 0.0000        0.0030           0.0000  
pop_exposures                     0.0000        0.0055           0.0000  
pop_engaged_rate                  0.0000        0.0000           0.0000  
pop_history                       0.0000        0.0032           0.0000  

[4 rows x 22 columns]

**Fingerprint.** Row counts and column sums of a few features per split (labels not used), so a build in another environment (Colab) can be compared with the local files exactly.

In [11]:
FINGERPRINT = ["pooled_product", "pooled_L3", "bm25", "title_jaccard", "pop_history_lag1d", "pop_engaged_rate",
               "days_since_last_action", "hist_len"]
pd.DataFrame({split: {"searches": table.session_id.nunique(), "rows": len(table),
                      **{c: float(np.nansum(table[c].to_numpy(dtype=np.float64))) for c in FINGERPRINT}}
              for split, table in tables.items()}).T

,searches,rows,pooled_product,pooled_L3,bm25,title_jaccard,pop_history_lag1d,pop_engaged_rate,days_since_last_action,hist_len
train_dev,"40,000.0000","3,573,362.0000","67,121.0000","19,816,644.0000","41,380,166.8717","271,424.1108","82,984,426.0000","82,592.0190","19,765,086.8476","539,236,069.0000"
train_rest,"93,831.0000","8,370,164.0000","160,017.0000","49,041,487.0000","96,866,115.5381","634,897.7219","190,822,932.0000","193,172.8358","46,394,638.1844","1,285,463,040.0000"
validation,"20,000.0000","1,781,573.0000","33,764.0000","9,864,983.0000","20,616,113.0741","135,274.0733","41,092,687.0000","41,154.2589","9,603,928.3902","272,368,032.0000"
test,"20,000.0000","1,784,913.0000","34,139.0000","10,087,548.0000","20,740,112.1586","135,571.9206","39,698,111.0000","41,127.1923","9,500,057.1910","270,911,837.0000"


In [12]:
train = tables["train_dev"]
train_sizes = searches.loc[BUILD["train_dev"], "n_candidates"]
train["score_random"] = 0.0
base = evaluate(train, "score_random", expected_sizes=train_sizes)
rows = {}
for feature in ["pooled_product", "pooled_shop_other", "click_0_7_product", "type_click_product", "bm25",
                "title_jaccard", "pop_exposures", "pop_engaged_rate", "pop_history", "pop_history_lag1d", "qrel_product", "linked_product",
                "qless_product"]:
    train["score"] = train[feature].fillna(-1)
    per = evaluate(train, "score", expected_sizes=train_sizes)
    rows[feature] = {"ndcg_at_10": per.ndcg.mean(), "gain_over_random": (per.ndcg - base.ndcg).mean()}
pd.DataFrame(rows).T

,ndcg_at_10,gain_over_random
pooled_product,0.1777,0.0944
pooled_shop_other,0.1348,0.0514
click_0_7_product,0.1521,0.0688
type_click_product,0.1521,0.0688
bm25,0.0913,0.0080
title_jaccard,0.0974,0.0140
pop_exposures,0.3146,0.2313
pop_engaged_rate,0.2516,0.1682
pop_history,0.3394,0.2560
pop_history_lag1d,0.3363,0.2529


**Popularity mechanism.** Split each training row's exposure count into exposures from other training searches with the **identical query** and from all **other queries**, and rank by each part alone. Also compare training searches whose query occurs elsewhere in training with those whose query does not.

In [13]:
same_query = []
for r in records:
    if r["split"] != "train_dev":
        continue
    counter = Counter()
    for session_id, wids in same_query_lists.get(r["query"], []):
        if session_id != r["session_id"]:
            counter.update(wids)
    same_query.append(np.array([counter[w] for w in r["candidates"]], dtype=float))
train["exp_same_query"] = np.concatenate(same_query)
train["exp_other_query"] = train.pop_exposures - train.exp_same_query
recurring = train.groupby("session_id").exp_same_query.transform("max") > 0
rows = {}
for name, score in {"all exposures": "pop_exposures", "identical query only": "exp_same_query",
                    "other queries only": "exp_other_query"}.items():
    train["score"] = train[score]
    per = evaluate(train, "score", expected_sizes=train_sizes)
    rows[name] = {"ndcg_at_10": per.ndcg.mean(), "gain_over_random": (per.ndcg - base.ndcg).mean()}
mechanism = pd.DataFrame(rows).T
mechanism["share_of_exposure_mass"] = [1.0, train.exp_same_query.sum() / train.pop_exposures.sum(),
                                       train.exp_other_query.sum() / train.pop_exposures.sum()]
train["score"] = train.pop_exposures
per = evaluate(train, "score", expected_sizes=train_sizes).join(
    recurring.groupby(train.session_id).first().rename("recurring_query"))
by_query = per.assign(random=base.ndcg).groupby("recurring_query")[["ndcg", "random"]].agg(["mean", "size"])
display(mechanism)
by_query

,ndcg_at_10,gain_over_random,share_of_exposure_mass
all exposures,0.3146,0.2313,1.0000
identical query only,0.1783,0.0950,0.1215
other queries only,0.3092,0.2259,0.8785


ndcg        random       
                  mean   size   mean   size
recurring_query                            
False           0.2818  22602 0.0994  22602
True            0.3586  16895 0.0618  16895

## Summary (E13, full build E21)

- **Feature set v3:** 97 features in 13 groups for 40,000 training (dev) searches (3,573,362 rows) and 20,000 validation searches (1,781,573 rows); `data/processed/features_v3_*.parquet`, groups in `features_v3_meta.json`. Changes from v2: `n_candidates` removed; `pop_history_lag1d` (deployable popularity) added and separated from same-day popularity; metadata coverage per history block (H5) added.
- **Checks passed:** complete lists, block consistency, permutation test, toy checks; training and validation distributions match (lagged history popularity zero share 60.4% vs 60.7%; largest zero-share gap of any feature 0.7 points).
- **Deployable popularity costs little:** on training rows lagged history popularity reaches NDCG@10 0.336 vs 0.339 unlagged, as in both audits (E12). All other single-signal values reproduce v2.
- **Build:** features are written into one pre-allocated float32 block per split; the earlier per-search DataFrames ran out of memory twice (16 GB) and took two hours. Now about 15 minutes for the features, 25 minutes in total.
- Searches with a clamped final gap are still included here; they are flagged in `search_table.parquet` and excluded from training and the main evaluation downstream (E13).

- **Full build on Colab (E21, 28 Oct, high-RAM CPU runtime):** all four splits, 173,831 searches: `train_dev` 3,573,362 rows, `train_rest` (93,831 searches) 8,370,164 rows, `validation` 1,781,573, `test` 1,784,913; features 18 minutes. The fingerprint (row counts and column sums) of `train_dev` and `validation` equals the local build exactly, so the notebook is reproducible on Colab. `train_rest` and `test` distributions match `train_dev` (largest zero-share gap 0.5 and 0.7 points). Test features are not evaluated until the final run.